# Quantitative Study 03: Survivorship Bias Audit & PCA Factor Decay
**Author**: Quantitative Research Desk  
**Objective**: Quantify the impact of survivorship bias using Point-in-Time Nifty 500 index reconstitutions, and demonstrate 10-tranche PCA residual momentum extraction.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from equity_bt.pit_universe import PointInTimeUniverse
from equity_bt.pca_factor_model import PCAResidualMomentumEngine, PCAStrategyConfig

### 1. Auditing Survivorship Bias in Nifty 500

In [ ]:
pit_json = "../sample_data/pit/nifty500_pit.json"
pit = PointInTimeUniverse(pit_json)

# Reconstruct universe on 2018-01-01
u_2018 = pit.get_universe_on_date("2018-01-01")
u_2024 = pit.get_universe_on_date("2024-01-01")

audit = PointInTimeUniverse.audit_survivorship_bias(static_universe=u_2024, pit_universe=u_2018)
print("--- Survivorship Bias Audit (2018 vs 2024 Constituents) ---")
print(f"Total Historical 2018 Constituents: {audit['total_pit_count']}")
print(f"Current Constituents Overlap: {audit['overlap_count']} ({audit['overlap_pct']:.1f}%)")
print(f"Survivorship-Biased Names Count: {audit['survivor_bias_stocks_count']}")
print(f"Sample Excluded / Delisted Stocks Recovered: {audit['historical_excluded_sample'][:8]}")

### 2. PCA Factor Decomposition (Stripping Market & Sector Betas)

In [ ]:
# Synthetic multi-asset returns matrix simulating Nifty 500 cross-section
np.random.seed(42)
dates = pd.date_range("2024-01-01", periods=150, freq="B")
tickers = [f"STOCK_{i:03d}" for i in range(50)]

# Generate market factor + idiosyncratic residuals
market_factor = np.random.normal(0.0005, 0.012, len(dates))
rets_data = {}
for sym in tickers:
    beta = np.random.uniform(0.6, 1.4)
    idio = np.random.normal(0.0, 0.018, len(dates))
    rets_data[sym] = beta * market_factor + idio

returns_df = pd.DataFrame(rets_data, index=dates)

engine = PCAResidualMomentumEngine(PCAStrategyConfig(n_components=2))
residuals = engine.extract_residual_returns(returns_df, n_components=2)

print(f"Raw Returns Cross-Section Variance: {returns_df.values.var():.6f}")
print(f"Residual Returns Cross-Section Variance: {residuals.values.var():.6f}")
print("Successfully isolated idiosyncratic alpha from systematic beta.")